In [ ]:
from glob import glob

for g in glob('../data/*.pdf'):
    print(g)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter



def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunk_overlap=100):
    """
    주어진 PDF 파일을 읽고 텍스트를 분할합니다.
    
    매개변수:
        pdf_path (str): PDF파일의 경로.
        chunk_size (int, 선택적): 각 텍스트 청크의 크기. 기본값은 1000입니다.
        chunk_overlap (int, 선택적): 청크 간의 중첩 크기. 기본값은 100입니다.

    반환값:
        list: 분할된 텍스트 청크의 리스트.
    """
    return(f"PDF: {pdf_path}------------------------------")

    pdf_loader = PyPDFLoader(pdf_path)
    data_from_pdf = pdf_loader.load()

    text_spliter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )

    splits = text_spliter.split_documnets(data_from_pdf)

    print(f"Number of splits: {len(splits)}\n")

    return splits

In [ ]:
    from langchain_openai import OpenAIEmbeddings
    from langchain_chroma import Chroma
    import os



    # vectorstore 설정
    embedding = OpenAIEmbeddings(model='text-embedding-3-large')

    persist_directory = '../chroma_store'

    if os.path.exists(persist_directory):
        print("Loading existing Chroma store")
        vectorstore = Chroma(
            persist_directory=persist_directory,
            embedding_function=embedding,
        )

    else:
        print("Creating new Chroma store")

        vectorstore = None

        if g in glob('../data/*.pdf'):
            chunks = read_pdf_and_split_text(g)
            # 100개씩 나눠서 저장
            for i in range(0, len(chunks), 100):
                if vectorstore is None:
                    vectorstore = Chroma.from_documents(
                        documents=chunks[i:i+100],
                        embedding=embedding,
                        persist_directory=persist_directory,
                    )
                else:
                    vectorstore.add_documents(
                        documents=chunks[i:i+100]
                    )

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs = {'k': 5})

chunks = retriever.invoke('서울 온실가스 저감 계획')

for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content)

In [ ]:
from langchain_openai import ChatOpenAI


model = ChatOpenAI(model='gpt-4o-mini')


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from typing import Literal
from pydantic import BaseModel, Field



class RouteQuery(BaseModel):
    """
    사용자 쿼리를 가장 관련성이 높은 데이터 소스로 라우팅합니다."""

    datasource: Literal["vectorstore", "casual_talk"] = Field(
        ...,
        description="""
        사용자 질문에 따라 casual_talk 또는 vectorstore로 라우팅합니다.
        - casual_talk: 일상 대화를 위한 데이터 소스. 사용자가 일상적인 질문을 할 때 사용합니다.
        - vectorstore: 사용자 질문에 답하기 위해 RAG로 vectorstore 검색이 필요한 경우 사용합니다.
        """,
    )

In [ ]:
structured_llm_router = model.with_structured_output(RouteQuery)

router_system = """
당신은 사용자의 질문을 vectorstore 또는 casual_talk으로 라우팅하는 전문가입니다.
- vectorstore에는 서울, 뉴욕의 발전계획과 관련된 문서가 포함되어 있습니다. 이 주제에 대한 질문에는 vectorstore를 사용하십시오.
- 사용자의 질문이 일상 대화에 관련된 경우 casual_talk를 사용하십시오.
"""

# 시스템 메시지와 사용자의 줄믄을 포함하는 프롬프트 템플릿 생성
router_prompt = ChatPromptTemplate.from_messages([
    ("system", router_system),
    ("human", "{question}"),
])

question_router = router_prompt | structured_llm_router

In [ ]:
print(
    question_router.invoke({
        "question" : "서울 온실가스 저감 계획은 무엇인가요?"
    })
)

print(question_router.invoke({"question": "잘 지냈어?"}))

In [ ]:
from langchain_core.prompts import PromptTemplate



class GradeDocuments(BaseModel):
    """검색된 문서가 질문과 관련성이 있는지 yes 또는 no로 평가합니다."""

    binary_score: Literal["yes", "no"] = Field(
        description="문서가 질문과 관련이 있는지 여부를 'yes' 또는 'no'로 평가합니다."
    )

structured_llm_grader = model.with_structured_output(GradeDocuments)

In [ ]:
grader_prompt = PromptTemplate.from_template("""
당신은 검색된 문서가 사용자 질문과 관련이 있는지 평가하는 평가자입니다. \n
문서에 사용자 질문과 관련된 키워드 또는 의미가 포함되어 있으면, 해당 문서를 관련성이 있다고 평가하십시오. \n
엄격한 테스트가 필요하지 않습니다. 목표는 잘못된 검색 결과를 걸러내는 것입니다. \n
문서가 질문과 관련이 있는지 여부를 나타내기 위해 'yes' 또는 'no'로 이진 점수를 부여하십시오.                                                                                                                   

Retrieved document: \n {document} \n\n
User question: {question}
""")

retrieval_grader = grader_prompt | structured_llm_grader
question = "서울시 자율주행 관련 계획"
documents = retriever.invoke(question)

for doc in documents:
    print(doc)

In [ ]:
filtered_docs = []

for i, doc in enumerate(documents):
    print(f"Document {i+1}:")
    is_relevant = retrieval_grader.invoke({"question":question, "document": doc.page_content})
    print(is_relevant)
    print(doc.page_content[:200])
    print("===================================\n\n")

    if is_relevant.binary_score == "yes":
        filtered_docs.append(doc)

print(f"Filtered documnets: {len(filtered_docs)}")

In [ ]:
rag_generate_system = """
너는 사용자의 질문에 대해 주어진 context에 기반하여 답변하는 도시 계획 전문가이다.
주어진 context는 vectorstore에서 검색된 결과이다.
주어진 context를 기반으로 사용자의 question에 대해 답변하라.

======================
question: {question}
context: {context}
"""

rag_prompt = PromptTemplate(
    input_variables=["question", "context"],
    template=rag_generate_system
)

rag_chain = rag_prompt | model

question = "서울시 자율주행 관련 계획"

rag_chain.invoke({"question":question, "context": filtered_docs})

In [ ]:
from typing import List
from typing_extensions import TypedDict



class GraphState(TypedDict):
    question: str
    generation: str
    documents: List[str]



In [ ]:
def route_question(state):
    """
    사용자 질문을 vectorstore 또는 casual_talk로 라우팅합니다.

    Args:
        state (dict): 현재 graph state

    return:
        state (dict): 라우팅된 데잉터 소스와 사용자 질문을 포함하는 새로운 graph state
    """

    print('-----ROUTE------')
    question = state["qeustion"]
    route = question_router.invoke({"question":question})



    print(f"---Routing to {route.datasource}---")
    return route.datasource

In [ ]:
def retrieve(state):
    """
    vectorstore에서 질문에 대한 문서를 검색합니다.

    Args:
        state (dict): 현재 graph state

    Return:
        state(dict): 검색된 문서와 사용자 질문을 포함하는 새로운 graph state
    """

    print('-----RETRIEVE-----')
    question = state["question"]

    documents = retriever.invoke(question)
    return {"documents":documents, "question":question}

In [ ]:
def grade_documents(state):
    """
    검색된 문서를 평가하여 질문과 관련성이 있는지 확입합니다.
    
    Args:
        state(dict): 현재 graph state
        
    return:
        state(dict): 관련성이 있는 문서와 사용자 질문을 포함하는 새로운 graph state
    """

    print('------GRADE------')
    question = state['question']
    documents = state['documents']
    filtered_docs = []

    for i, doc in enumerate(documents):
        is_relevant = retrieval_grader.invoke({"question": question, "document": doc.page_content})

        if is_relevant.binary_score == "yes":
            filtered_docs.append(doc)

    return {"documents": filtered_docs, "question": question}

In [ ]:
def generate(state):
    """
    LLM을 사용하여 문서와 사용자 질문에 대한 답변을 생성합니다.

    Args:
        state(dict): 현재 graph state

    Return:
        state(dict): LLM 생성 결과와 사용자 질문을 포함하는 새로운 graph state
    """

    print('------GENERATE------')
    question = state['question']
    documents = state['documents']
    generation = rag.chain.invoke({"question": question, "context": documents})
    return {
        "documents":documents,
        "question":question,
        "generation": generation
    }

In [ ]:
def casual_talk(state):
    """
    일상 대화를 위한 답변을 생성합니다.

    Args:
        state(dict) : 현재 graph state

    return:
        state(dict) : 일상 대화 결과와 사용자 질문을 포함하는 새로운 graph state
    """
    print('------CASUAL TALK------')
    question = state['question']
    generation = model.invoke(question)
    return {
        "question": question,
        "generation": generation
    }

In [ ]:
from langgraph.graph import START, StateGraph, END



workflow = StateGraph(GraphState)

In [ ]:
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("casual_talk", casual_talk)


In [ ]:
workflow.add_conditional_edges(
    START,
    route_question,
    {
        "vectorstore": "retrieve",
        "casual_talk": "casual_talk"
    }
)

workflow.add_edge("casual_talk", END)
workflow.add_edge("retrieve", grade_documents)
workflow.add_edge("grade_documents", generate)
workflow.add_edge("generate", END)

app = workflow.compile()
